# Modelo inicial de evaluación 1 de TAVI


In [1]:
import sys, os, platform, subprocess, textwrap
print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("SO:", platform.platform())
print("Carpeta de trabajo:", os.getcwd())

Python: 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Ejecutable: d:\Carpeta\Escritorio\Eval-1-TAVI\.venv\Scripts\python.exe
SO: Windows-11-10.0.26200-SP0
Carpeta de trabajo: d:\Carpeta\Escritorio\Eval-1-TAVI


In [2]:
# Instalacion de Langchain, creador de pdf, entre otros
%pip install -q -r requirements.txt
%pip install fpdf2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Importar Langchain y componentes
import importlib, langchain, langchain_core, langchain_community
print("LangChain version:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core version:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Community version:", getattr(langchain_community, "__version__", "unknown"))
print("OK: importaciones realizadas.")

LangChain version: 0.3.28
LangChain Core version: 0.3.84
LangChain Community version: 0.3.31
OK: importaciones realizadas.


In [10]:
#importar chatllama
import os
from langchain_community.chat_models import ChatLlamaCpp
from huggingface_hub import hf_hub_download

# 1. Configuracion del modelo (Siguiendo el Ejemplo 1) 
model_name = "bartowski/Llama-3.2-3B-Instruct-GGUF"
model_file = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"

# 2. Descargar y obtener la ruta local 
# local_dir crea la carpeta automaticamente si no existe
model_path = hf_hub_download(
    repo_id=model_name,
    filename=model_file,
    local_dir="./modelos", 
)

# Linea de codigo agregada para que funcione llama
%pip install llama-cpp-python --only-binary :all: --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu


llm = ChatLlamaCpp(
    model_path=model_path,
    n_ctx=4096,          # Ventana de contexto (memoria)
    max_tokens=2048,     # para que escriba mas
    n_batch=16,          # Tokens procesados en paralelo
    n_threads=None,      # Numero de hilos del procesador a usar
    n_gpu_layers=-1,     # Aceleracion por hardware
    temperature=0.0,     # Cero para mayor adherencia al formato
    top_k=20,
    top_p=0.9,
    format="json",       # Obliga a responder con la estructura de datos para poder estructurar el CV 
)

print("Cerebro local cargado correctamente y listo para armar CVs.")

llama_model_loader: loaded meta data with 35 key-value pairs and 255 tensors from modelos\Llama-3.2-3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:                            general.license str              = llama3.2
llama_model_loader: - kv   7:             

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
Note: you may need to restart the kernel to use updated packages.


llama_model_loader: - kv  24:                      tokenizer.ggml.tokens arr[str,128256]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  25:                  tokenizer.ggml.token_type arr[i32,128256]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  26:                      tokenizer.ggml.merges arr[str,280147]  = ["Ġ Ġ", "Ġ ĠĠĠ", "ĠĠ ĠĠ", "...
llama_model_loader: - kv  27:                tokenizer.ggml.bos_token_id u32              = 128000
llama_model_loader: - kv  28:                tokenizer.ggml.eos_token_id u32              = 128009
llama_model_loader: - kv  29:                    tokenizer.chat_template str              = {{- bos_token }}\n{%- if custom_tools ...
llama_model_loader: - kv  30:               general.quantization_version u32              = 2
llama_model_loader: - kv  31:                      quantize.imatrix.file str              = /models_out/Llama-3.2-3B-Instruct-GGU...
llama_model_loader: - kv  32:                   quan

Cerebro local cargado correctamente y listo para armar CVs.


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'general.name': 'Llama 3.2 3B Instruct', 'general.architecture': 'llama', 'general.type': 'model', 'llama.block_count': '28', 'general.basename': 'Llama-3.2', 'general.finetune': 'Instruct', 'general.size_label': '3B', 'general.license': 'llama3.2', 'llama.context_length': '131072', 'llama.embedding_length': '3072', 'llama.feed_forward_length': '8192', 'llama.attention.head_count': '24', 'tokenizer.ggml.eos_token_id': '128009', 'general.file_type': '15', 'llama.attention.head_count_kv': '8', 'llama.rope.freq_base': '500000.000000', 'quantize.imatrix.entries_count': '196', 'llama.attention.layer_norm_rms_epsilon': '0.000010', 'llama.attention.key_length': '128', 'llama.attention.value_length': '128', 'llama.vocab_size': '128256', 'llama.rope.dimension_count': '128', 'tokenizer.ggml.model': 'gpt2', 'tokenizer.ggml.pre': 'llama-bpe', 'general.quantization_vers

In [14]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

class Educacion(BaseModel):
    titulo: str = Field(description="Titulo, carrera o certificacion")
    institucion: str = Field(description="Universidad o institucion")
    periodo: str = Field(description="Ej: 2015-2019 o 'En proceso de titulacion'")

class Experiencia(BaseModel):
    cargo_adaptado: str = Field(description="Cargo y empresa (ej: Analista de Datos, Empresa X)")
    periodo: str = Field(description="Fecha inicio - Fecha termino")
    descripcion_breve: str = Field(description="Descripcion breve del rol (2 lineas maximo)")
    logros_clave: List[str] = Field(description="Logros usando: Verbo + Accion + Impacto + Metrica")

class CVResultado(BaseModel):
    nombre_completo: str = Field(description="Nombre completo")
    profesion: str = Field(description="Profesion actual o titulo profesional")
    contacto: str = Field(description="Celular | Email | LinkedIn")
    resumen_profesional: str = Field(description="Maximo 5 lineas. Propuesta de valor y areas de interes.")
    experiencias: List[Experiencia]
    educacion: List[Educacion]
    habilidades: List[str] = Field(description="Keywords tecnicas, idiomas (con nivel) y certificaciones")

parser = JsonOutputParser(pydantic_object=CVResultado)

# ==========================================
# 2. EL PROMPT EXPERTO (Basado en tu Rubrica)
# ==========================================
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un Coach experto en empleabilidad y formatos ATS. Tu mision es tomar datos desordenados y crear un CV de Alto Impacto.
APLICA ESTAS REGLAS ESTRICTAMENTE:
1. RESUMEN PROFESIONAL: Maximo 5 lineas. Incluye la propuesta de valor y las areas de interes/disciplina donde el usuario quiere desarrollarse.
2. LOGROS EN LUGAR DE TAREAS: Transforma la experiencia bruta en logros. No escribas funciones. Usa la formula: Verbo + Accion + Impacto + Metrica.
   - Usa verbos como: Reducir, Optimizar, Incrementar, Fidelizar, Implementar.
   - Ejemplo: "Optimice procesos de venta, reduciendo tiempos en un 20%".
3. EDUCACION: Si el usuario dice "presente", indica claramente si es "Estudiante" o "En proceso de titulacion". Omite colegio basico/medio.
4. HABILIDADES: Destaca keywords tecnicas. Si hay idiomas, especifica el nivel (ej: Ingles avanzado).
5. FORMATO: Devuelve SOLO un objeto JSON valido, sin conversar.
6. SI NO PUEDES CUMPLIR EL FORMATO, DEVUELVE UN JSON VALIDO CON LAS CLAVES REQUERIDAS Y LISTAS VACIAS.
7. DEBES COMPLETAR experiencias y educacion si hay datos en el input, aunque sean breves.
\nINSTRUCCIONES DE FORMATO:\n{format_instructions}"""),
    ("human", """Cargo Objetivo: {cargo_objetivo}
Nombre: {nombre}
Contacto: {contacto}
Profesion actual: {profesion}
Formacion academica: {educacion_bruta}
Habilidades clave: {habilidades}
Experiencia laboral: {experiencia_bruta}
\nRecuerda: responde SOLO JSON valido con las claves exactas.""")
])

# Prompt de reparacion si el JSON sale mal
repair_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un validador de JSON estricto.
Reescribe el contenido entregado para que sea SOLO un JSON valido que cumpla el schema.
No inventes datos nuevos, solo reestructura y corrige el formato. Sin texto adicional."""),
    ("human", """Schema: {format_instructions}
Contenido a reparar: {contenido}""")
])

cadena_cv = prompt | llm
cadena_cv_repair = repair_prompt | llm
print("Fase 3 lista: Cadena y moldes Pydantic optimizados.")

Fase 3 lista: Cadena y moldes Pydantic optimizados.


In [5]:
from fpdf import FPDF
from fpdf.enums import XPos, YPos

def generar_pdf_cv(datos_cv, nombre_archivo="CV_Optimizado_IA.pdf"):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    w_total = 180
    
    # Funcion sanitizadora segura
    def txt(diccionario, clave, default=""):
        v = diccionario.get(clave, default)
        if isinstance(v, dict):
            return str(next(iter(v.values()))) if v else default
        if isinstance(v, list):
            return " ".join([str(i) for i in v])
        return str(v).replace("•", "-").replace("\u2022", "-")

    # ENCABEZADO
    pdf.set_font("helvetica", "B", 16)
    pdf.cell(w_total, 8, txt(datos_cv, "nombre_completo", "SIN NOMBRE").upper(), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    
    pdf.set_font("helvetica", "B", 12)
    pdf.cell(w_total, 6, txt(datos_cv, "profesion", "SIN PROFESION").upper(), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    
    pdf.set_font("helvetica", "", 10)
    pdf.cell(w_total, 6, txt(datos_cv, "contacto", "Sin contacto"), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    pdf.cell(w_total, 8, "", new_x=XPos.LMARGIN, new_y=YPos.NEXT) 
    
    def titulo(texto):
        pdf.set_font("helvetica", "B", 12)
        pdf.cell(w_total, 8, texto, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(pdf.get_x(), pdf.get_y(), pdf.get_x() + w_total, pdf.get_y())
        pdf.cell(w_total, 3, "", new_x=XPos.LMARGIN, new_y=YPos.NEXT)

    # RESUMEN
    titulo("Resumen Profesional")
    pdf.set_font("helvetica", "", 10)
    pdf.multi_cell(w_total, 5, txt(datos_cv, "resumen_profesional", "Sin resumen."))
    pdf.cell(w_total, 5, "", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    
    # EXPERIENCIA
    titulo("Experiencia Laboral")
    experiencias = datos_cv.get("experiencias", []) if isinstance(datos_cv.get("experiencias"), list) else []
    for exp in experiencias:
        if not isinstance(exp, dict):
            continue
        cargo = txt(exp, "cargo_adaptado", txt(exp, "cargo_empresa_pais", "Experiencia"))
        pdf.set_font("helvetica", "B", 10)
        pdf.cell(130, 6, cargo)
        pdf.set_font("helvetica", "", 10)
        pdf.cell(50, 6, txt(exp, "periodo"), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="R")
        
        descripcion = txt(exp, "descripcion_breve", "")
        if descripcion:
            pdf.multi_cell(w_total, 5, descripcion)

        logros = exp.get("logros_clave", [])
        logros = logros if isinstance(logros, list) else [logros]
        for logro in logros:
            texto_logro = txt({"l": logro}, "l")
            pdf.multi_cell(w_total, 5, f"- {texto_logro}")
        pdf.cell(w_total, 3, "", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
    # EDUCACION
    titulo("Formacion Educativa")
    educacion = datos_cv.get("educacion", []) if isinstance(datos_cv.get("educacion"), list) else []
    for edu in educacion:
        if not isinstance(edu, dict):
            continue
        pdf.set_font("helvetica", "B", 10)
        pdf.cell(130, 6, txt(edu, "titulo"))
        pdf.set_font("helvetica", "", 10)
        pdf.cell(50, 6, txt(edu, "periodo"), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="R")
        pdf.multi_cell(w_total, 5, txt(edu, "institucion"))
        pdf.cell(w_total, 2, "", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
    # HABILIDADES
    titulo("Habilidades")
    pdf.set_font("helvetica", "", 10)
    habilidades = datos_cv.get("habilidades")
    if not isinstance(habilidades, list):
        habilidades = datos_cv.get("habilidades_y_certificaciones", []) if isinstance(datos_cv.get("habilidades_y_certificaciones"), list) else []
    for hab in habilidades:
        pdf.multi_cell(w_total, 5, f"- {txt({'h': hab}, 'h')}")
        
    pdf.output(nombre_archivo)

print("Fase 4: Motor de PDF blindado.")

Fase 4: Motor de PDF blindado.


In [15]:
import json

print("=== Generador de CV y Debugger (ATS Ready) ===")
print("Escribe 'salir' para cancelar.\n")

while True:
    nombre = input("1. Nombre completo: ")
    if nombre.lower() == 'salir': break
    contacto = input("2. Contacto (Email/Teléfono): ")
    profesion = input("3. Profesión actual: ")
    cargo_objetivo = input("4. Cargo al que postulas: ")
    experiencia_bruta = input("5. Experiencia laboral: ")
    educacion_bruta = input("6. Formación académica: ")
    habilidades = input("7. Habilidades clave: ")

    print("\n⏳ Procesando con IA (Esto tomará unos segundos)...")
    
    try:
        payload = {
            "cargo_objetivo": cargo_objetivo,
            "nombre": nombre,
            "contacto": contacto,
            "profesion": profesion,
            "educacion_bruta": educacion_bruta,
            "habilidades": habilidades,
            "experiencia_bruta": experiencia_bruta,
            "format_instructions": parser.get_format_instructions(),
        }

        # 1) Genera texto crudo con el LLM
        raw_msg = cadena_cv.invoke(payload)
        raw_text = raw_msg.content if hasattr(raw_msg, "content") else str(raw_msg)

        # 2) Intenta parsear JSON; si falla, repara y vuelve a parsear
        try:
            resultado = parser.parse(raw_text)
        except Exception:
            repair_msg = cadena_cv_repair.invoke({
                "format_instructions": parser.get_format_instructions(),
                "contenido": raw_text,
            })
            repair_text = repair_msg.content if hasattr(repair_msg, "content") else str(repair_msg)
            resultado = parser.parse(repair_text)

        if not isinstance(resultado, dict):
            raise ValueError("La salida del parser no es un diccionario JSON")

        # Fallbacks para no perder lo ingresado por usuario
        resultado["nombre_completo"] = resultado.get("nombre_completo") or nombre
        resultado["contacto"] = resultado.get("contacto") or contacto
        resultado["profesion"] = resultado.get("profesion") or profesion

        if not resultado.get("resumen_profesional"):
            resultado["resumen_profesional"] = f"{profesion}. Interes en {cargo_objetivo}."

        if not isinstance(resultado.get("experiencias"), list):
            resultado["experiencias"] = []

        if not isinstance(resultado.get("educacion"), list):
            resultado["educacion"] = []

        if not isinstance(resultado.get("habilidades"), list):
            habilidades_lista = [h.strip() for h in habilidades.split(",") if h.strip()]
            resultado["habilidades"] = habilidades_lista

        # Fallbacks: si la IA no lleno experiencia o educacion, usa el input
        if not resultado["experiencias"] and experiencia_bruta.strip():
            resultado["experiencias"] = [
                {
                    "cargo_adaptado": experiencia_bruta.strip(),
                    "periodo": "",
                    "descripcion_breve": "",
                    "logros_clave": [],
                }
            ]

        educacion_txt = educacion_bruta.strip()
        if educacion_txt and ("colegio" not in educacion_txt.lower()) and not resultado["educacion"]:
            resultado["educacion"] = [
                {
                    "titulo": educacion_txt,
                    "institucion": "",
                    "periodo": "",
                }
            ]

        # --- MODO DEBUG: IMPRIMIR JSON ---
        print("\n" + "="*50)
        print("🛠️ JSON INTERNO GENERADO POR LA IA:")
        print(json.dumps(resultado, indent=4, ensure_ascii=False))
        print("="*50)

        # --- GENERAR PDF ---
        nombre_limpio = nombre.replace(' ', '_') if nombre.strip() else "Usuario"
        archivo_pdf = f"CV_{nombre_limpio}.pdf"
        generar_pdf_cv(resultado, nombre_archivo=archivo_pdf)
        
        print(f"\n✅ ¡PDF generado con éxito: {archivo_pdf}!")

    except Exception as e:
        print(f"\n❌ Error fatal: La IA no pudo estructurar los datos correctamente. Detalle: {e}")
        print("Intenta ser más específico o usar el modelo Llama 3.2 3B.")
    
    if input("\n¿Deseas crear otro CV? (s/n): ").lower() != 's':
        break

=== Generador de CV y Debugger (ATS Ready) ===
Escribe 'salir' para cancelar.


⏳ Procesando con IA (Esto tomará unos segundos)...


Llama.generate: 331 prefix-match hit, remaining 877 prompt tokens to eval
llama_perf_context_print:        load time =  100578.42 ms
llama_perf_context_print: prompt eval time =  103975.66 ms /   877 tokens (  118.56 ms per token,     8.43 tokens per second)
llama_perf_context_print:        eval time =  129169.48 ms /   261 runs   (  494.90 ms per token,     2.02 tokens per second)
llama_perf_context_print:       total time =  234451.09 ms /  1138 tokens
llama_perf_context_print:    graphs reused =        309



🛠️ JSON INTERNO GENERADO POR LA IA:
{
    "Educacion": [
        {
            "titulo": "Colegio Melipilla",
            "institucion": "Colegio Melipilla",
            "periodo": "2006-2020"
        }
    ],
    "Experiencia": [
        {
            "cargo_adaptado": "Ayudante de estadística",
            "periodo": "2023-2024",
            "descripcion_breve": "Apoyo en análisis",
            "logros_clave": [
                "Apoyo en análisis"
            ]
        },
        {
            "cargo_adaptado": "Ayudante de estadística",
            "periodo": "2023-2024",
            "descripcion_breve": "Apoyo en análisis",
            "logros_clave": [
                "Apoyo en análisis"
            ]
        }
    ],
    "Habilidades": [
        "Excel",
        "Python",
        "SQL"
    ],
    "nombre_completo": "Thomas",
    "profesion": "Estudiante universitario",
    "contacto": "+56975432198",
    "resumen_profesional": "Propuesta de valor y areas de interes. Estudiante u